# L9: Function Calling

> 掌握 LLM Function Calling 技术，让 Agent 能够调用外部工具

In [ ]:
# === 环境设置 ===
import sys
import os
import json
import asyncio
from pathlib import Path

# 自动找到项目根目录
current_path = Path(os.getcwd())
project_path = None

for path in [current_path] + list(current_path.parents):
    if (path / "backend").exists():
        project_path = str(path)
        break

if project_path and project_path not in sys.path:
    sys.path.insert(0, project_path)

# 加载 .env
env_file = Path(project_path) / ".env" if project_path else None
if env_file and env_file.exists():
    with open(env_file, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line and not line.startswith("#") and "=" in line:
                key, value = line.split("=", 1)
                os.environ[key.strip()] = value.strip()

print(f"项目路径: {project_path}")
print(f"Python 版本: {sys.version.split()[0]}")

# 验证模块导入
try:
    from backend.llm import LLMFactory, Message
    from backend.tools.registry import ToolRegistry
    print("✓ 模块导入成功")
except ImportError as e:
    print(f"✗ 模块导入失败: {e}")

## 9.1 什么是 Function Calling？

### 定义

**Function Calling** 是 LLM 的一项能力，可以根据用户意图，**输出结构化的工具调用信息**，而不是生成文本。

### 核心价值

```
没有 Function Calling:
  用户: "北京现在几点？"
  LLM: "我无法知道当前时间..." (因为训练数据不是实时的)

有 Function Calling:
  用户: "北京现在几点？"
  LLM: get_time(location="Beijing")
  系统: 调用函数 → 返回时间 → LLM 组织回答
  LLM: "北京现在是 2024-01-15 14:30。"
```

### 工作流程

```
┌─────────────────────────────────────────────────────┐
│  1. 用户提问                                         │
│     "帮我查一下北京的天气"                           │
└──────────────────┬──────────────────────────────────┘
                   │
                   ▼
┌─────────────────────────────────────────────────────┐
│  2. LLM 分析意图 + 工具定义                          │
│     → 需要调用 get_weather(city="北京")            │
└──────────────────┬──────────────────────────────────┘
                   │
                   ▼
┌─────────────────────────────────────────────────────┐
│  3. 系统执行函数                                     │
│     → 调用天气 API → 获取结果                        │
└──────────────────┬──────────────────────────────────┘
                   │
                   ▼
┌─────────────────────────────────────────────────────┐
│  4. 结果返回给 LLM                                   │
│     → tool_result = {"temp": 5, "condition": "晴"} │
└──────────────────┬──────────────────────────────────┘
                   │
                   ▼
┌─────────────────────────────────────────────────────┐
│  5. LLM 生成最终回答                                 │
│     "北京今天是晴天，气温5度。"                      │
└─────────────────────────────────────────────────────┘
```

## 9.2 工具定义格式

### OpenAI Function Calling 格式（标准）

In [ ]:
# 工具定义示例
tool_definition = {
    "type": "function",
    "function": {
        "name": "get_weather",
        "description": "获取指定城市的天气信息",
        "parameters": {
            "type": "object",
            "properties": {
                "city": {
                    "type": "string",
                    "description": "城市名称，如：北京、上海"
                },
                "unit": {
                    "type": "string",
                    "enum": ["celsius", "fahrenheit"],
                    "description": "温度单位"
                }
            },
            "required": ["city"]
        }
    }
}

print("=== 工具定义格式 ===\n")
print(json.dumps(tool_definition, indent=2, ensure_ascii=False))

print("\n=== 关键字段说明 ===\n")
print("type: 固定为 'function'")
print("function.name: 函数名称（需与实际函数对应）")
print("function.description: 功能描述（LLM 据此选择何时调用）")
print("function.parameters: 参数定义（JSON Schema 格式）")
print("  - properties: 参数列表")
print("  - required: 必填参数")

### 项目中的工具定义

In [ ]:
# 查看项目中已注册的工具
from backend.tools.registry import ToolRegistry

print("=== 项目中已注册的工具 ===\n")
tools = ToolRegistry.get_all_tools()

for name, tool in tools.items():
    print(f"工具名称: {name}")
    print(f"描述: {tool.description}")
    print(f"参数: {[p.name for p in tool.parameters]}")
    print()

### 查看工具的 LLM 定义格式

In [ ]:
# 获取工具的 LLM 格式定义
llm_tools = ToolRegistry.get_llm_tool_definitions()

print("=== 工具的 LLM 定义格式 ===\n")
if llm_tools:
    # 展示第一个工具的定义
    print(json.dumps(llm_tools[0], indent=2, ensure_ascii=False))

## 9.3 Function Calling 调用流程

### 完整示例

In [ ]:
async def function_calling_demo():
    """Function Calling 完整示例"""
    from pathlib import Path
    
    # 获取 API Key
    def get_deepseek_key():
        project_path = Path(os.getcwd())
        for path in [project_path] + list(project_path.parents):
            env_file = path / ".env"
            if env_file.exists():
                with open(env_file, encoding="utf-8") as f:
                    for line in f:
                        line = line.strip()
                        if line.startswith("DEEPSEEK_API_KEY="):
                            return line.split("=", 1)[1].strip()
        return ""
    
    api_key = get_deepseek_key()
    if not api_key:
        print("⚠️  请设置 DEEPSEEK_API_KEY")
        return
    
    llm = LLMFactory.create("deepseek", api_key=api_key)
    
    # 定义工具
    tools = ToolRegistry.get_llm_tool_definitions()
    
    # 用户问题
    user_question = "帮我计算 123 * 456"
    
    print(f"=== 用户问题: {user_question} ===\n")
    
    # 第一步：LLM 决定是否调用工具
    response = await llm.chat(
        [Message(role="user", content=user_question)],
        tools=tools
    )
    
    # 检查是否有工具调用
    if response.tool_calls:
        print(f"=== LLM 决定调用工具 ===\n")
        for tool_call in response.tool_calls:
            print(f"工具: {tool_call.function.name}")
            print(f"参数: {tool_call.function.arguments}")
        
        # 第二步：执行工具
        print(f"\n=== 执行工具 ===\n")
        from backend.agents.base import BaseAgent
        
        # 模拟工具执行（实际在 BaseAgent.act 中）
        tool_results = []
        for tool_call in response.tool_calls:
            tool_name = tool_call.function.name
            tool_args = json.loads(tool_call.function.arguments)
            
            tool = ToolRegistry.get(tool_name)
            if tool:
                result = await tool.aexecute(**tool_args)
                tool_results.append({
                    "tool": tool_name,
                    "success": result.success,
                    "data": result.data
                })
                print(f"{tool_name} 结果: {result.data}")
        
        # 第三步：将结果返回给 LLM
        print(f"\n=== 生成最终回答 ===\n")
        followup_messages = [
            Message(role="user", content=user_question),
            Message(
                role="assistant",
                content=response.content,
                tool_calls=response.tool_calls
            ),
            Message(
                role="tool",
                content=json.dumps(tool_results, ensure_ascii=False),
                tool_call_id=response.tool_calls[0].id
            )
        ]
        
        final_response = await llm.chat(followup_messages)
        print(f"最终回答: {final_response.content}")
    else:
        print("LLM 没有调用工具，直接回答:")
        print(response.content)

# 运行（需要取消注释）
# await function_calling_demo()

## 9.4 项目中的 Function Calling 实现

In [ ]:
from backend.agents.base import BaseAgent
import inspect

# 查看 BaseAgent 中的 act 方法
print("=== BaseAgent.act 方法签名 ===\n")
print(inspect.signature(BaseAgent.act))

print("\n=== act 方法源码（部分）===\n")
source = inspect.getsource(BaseAgent.act)
print(source[:1000] + "...")

## 9.5 工具参数解析

### 从 LLM 返回到工具执行

In [ ]:
from backend.llm.models import ToolCall, FunctionDefinition

# 模拟 LLM 返回的工具调用
tool_call_example = ToolCall(
    id="call_123",
    type="function",
    function=FunctionDefinition(
        name="calculator",
        arguments='{"expression": "123 * 456"}'
    )
)

print("=== 工具调用示例 ===\n")
print(f"工具 ID: {tool_call_example.id}")
print(f"工具名称: {tool_call_example.function.name}")
print(f"参数 JSON: {tool_call_example.function.arguments}")

# 解析参数
args = json.loads(tool_call_example.function.arguments)
print(f"\n解析后参数: {args}")

# 获取工具
tool = ToolRegistry.get("calculator")
if tool:
    print(f"\n工具描述: {tool.description}")
    print(f"参数列表: {[p.name for p in tool.parameters]}")

## 9.6 并发工具调用

### 多工具同时调用

In [ ]:
async def parallel_tool_execution():
    """并发执行多个工具调用"""
    
    # 模拟 LLM 返回多个工具调用
    tool_calls = [
        ToolCall(
            id="call_1",
            type="function",
            function=FunctionDefinition(
                name="calculator",
                arguments='{"expression": "100 + 200"}'
            )
        ),
        ToolCall(
            id="call_2",
            type="function",
            function=FunctionDefinition(
                name="calculator",
                arguments='{"expression": "50 * 3"}'
            )
        ),
    ]
    
    print("=== 并发执行工具调用 ===\n")
    
    # 顺序执行
    import time
    
    start = time.time()
    for tc in tool_calls:
        args = json.loads(tc.function.arguments)
        tool = ToolRegistry.get(tc.function.name)
        if tool:
            result = await tool.aexecute(**args)
            print(f"{tc.function.name}({args}): {result.data}")
    sequential_time = time.time() - start
    print(f"\n顺序执行耗时: {sequential_time:.3f}秒")
    
    # 并发执行
    print("\n=== 并发执行 ===")
    
    async def execute_tool_call(tc):
        args = json.loads(tc.function.arguments)
        tool = ToolRegistry.get(tc.function.name)
        if tool:
            result = await tool.aexecute(**args)
            return {"call": tc.id, "result": result.data}
    
    start = time.time()
    results = await asyncio.gather(*[
        execute_tool_call(tc) for tc in tool_calls
    ])
    parallel_time = time.time() - start
    
    for r in results:
        print(f"{r['call']}: {r['result']}")
    print(f"\n并发执行耗时: {parallel_time:.3f}秒")
    print(f"加速比: {sequential_time/parallel_time:.2f}x")

await parallel_tool_execution()

## 9.7 Function Calling 最佳实践

In [ ]:
best_practices = {
    "工具描述": "清晰描述工具功能，帮助 LLM 正确选择",
    "参数描述": "每个参数都要有描述，说明取值范围",
    "必填字段": "required 明确指定必填参数",
    "枚举值": "使用 enum 限制参数取值范围",
    "错误处理": "工具执行失败要返回清晰错误信息",
    "结果格式": "返回结构化数据，便于 LLM 理解",
}

print("=== Function Calling 最佳实践 ===\n")
for key, value in best_practices.items():
    print(f"✓ {key}: {value}")

## 9.8 常见面试问题

**Q: 什么是 Function Calling？**
- A:
  - LLM 输出结构化的工具调用信息而非文本
  - 格式：`{"name": "func_name", "arguments": {...}}`
  - 让 LLM 能够调用外部 API 和函数

**Q: Function Calling 的完整流程是什么？**
- A:
  1. 定义工具（JSON Schema 格式）
  2. 发送请求（附带 tools 定义）
  3. LLM 返回 tool_calls
  4. 解析参数，执行函数
  5. 将结果返回给 LLM
  6. LLM 生成最终回答

**Q: 如何让 LLM 更准确地选择工具？**
- A:
  1. **清晰描述**: 工具描述要具体说明用途
  2. **参数描述**: 每个参数都要有详细说明
  3. **示例引导**: 在 system_prompt 中给出示例
  4. **限制数量**: 不要一次性提供太多工具

**Q: Function Calling 和 ReAct 有什么区别？**
- A:
  | | Function Calling | ReAct |
  |-|------------------|------|
  | **本质** | LLM 能力 | Agent 架构 |
  | **输出** | 结构化工具调用 | 文本+工具调用 |
  | **范围** | 单次工具选择 | 完整推理循环 |
  | **关系** | ReAct 使用 Function Calling | - |

**Q: 如何处理工具执行失败？**
- A:
  1. 返回清晰的错误信息给 LLM
  2. 让 LLM 决定是重试还是尝试其他方法
  3. 可以设置最大重试次数
  4. 记录错误日志用于监控

**Q: 如何实现工具调用链（一个工具的结果传给另一个）？**
- A:
  1. 通过多轮对话实现
  2. 将第一个工具的结果作为 context
  3. LLM 基于结果决定下一个工具
  4. ReAct 模式天然支持调用链

**Q: 一次可以调用多个工具吗？**
- A:
  - 是的，LLM 可以返回多个 tool_calls
  - 可以并发执行（如果工具之间无依赖）
  - 响应中的 tool_calls 是一个数组
  - 每个 tool_call 有唯一的 id

**Q: JSON Schema 在 Function Calling 中的作用？**
- A:
  1. 定义参数结构
  2. 指定参数类型
  3. 标记必填参数
  4. 限制枚举值
  5. LLM 据此生成正确格式的参数

---

## 总结

本课程学习了 Function Calling 的核心知识：

| 知识点 | 说明 |
|--------|------|
| **定义** | LLM 输出结构化工具调用而非文本 |
| **格式** | OpenAI Function Calling 格式 |
| **流程** | 定义→调用→执行→返回→生成 |
| **工具定义** | JSON Schema 格式 |
| **参数解析** | 从 JSON 到函数参数 |
| **并发调用** | 多工具同时执行 |
| **最佳实践** | 清晰描述、错误处理、结果格式 |

**第二阶段完成！** 下一步：L10 工具系统！